# Week 4: MNIST Evaluation and Tuning

This notebook evaluates the Logistic Regression baseline and small neural network trained from the Week 3 arrays. It reports accuracy, classification metrics, confusion matrices, tuning results, and sample predictions for the Month 1 report.

In [ ]:
from pathlib import Path
import sys
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

        CURRENT_DIR = Path.cwd()
        if CURRENT_DIR.name == 'notebooks':
            REPOSITORY_ROOT = CURRENT_DIR.parents[1]
        elif CURRENT_DIR.name == 'week-4':
            REPOSITORY_ROOT = CURRENT_DIR.parent
        else:
            REPOSITORY_ROOT = CURRENT_DIR
        WEEK4_ROOT = REPOSITORY_ROOT / 'week-4' if (REPOSITORY_ROOT / 'week-4').exists() else REPOSITORY_ROOT
sys.path.insert(0, str(WEEK4_ROOT / 'src'))
from train_mnist_model import load_week3_arrays, train_models, tune_neural_network

MODEL_DIR = WEEK4_ROOT / 'models'
x_train, x_test, y_train, y_test = load_week3_arrays()
print(f'Train images: {x_train.shape}; test images: {x_test.shape}')
print(f'Train labels: {y_train.shape}; test labels: {y_test.shape}')

## Train or load the comparison models

Run the training script first for a repeatable command-line workflow. This cell can also train the comparison models interactively when the saved files are not present.

In [ ]:
baseline_path = MODEL_DIR / 'baseline_logistic_regression.pkl'
network_path = MODEL_DIR / 'neural_network.pkl'
if baseline_path.exists() and network_path.exists():
    models = {'baseline': joblib.load(baseline_path), 'neural_network': joblib.load(network_path)}
    print('Loaded saved comparison models.')
else:
    models = train_models(x_train, y_train)
    print('Trained comparison models in the notebook.')

In [ ]:
results = {}
for name, model in models.items():
    predictions = model.predict(x_test)
    results[name] = {
        'predictions': predictions,
        'accuracy': accuracy_score(y_test, predictions),
        'report': classification_report(y_test, predictions, digits=4),
    }
    print(f'### {name} accuracy: {results[name]["accuracy"]:.4f}')
    print(results[name]['report'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (name, result) in zip(axes, results.items()):
    matrix = confusion_matrix(y_test, result['predictions'])
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axis)
    axis.set_title(f'{name.replace("_", " ").title()} confusion matrix')
    axis.set_xlabel('Predicted label')
    axis.set_ylabel('True label')
plt.tight_layout()
plt.show()

## Tune the stronger model

The comparison above identifies the stronger model by test accuracy. For this project, the small neural network is tuned by trying a few hidden-layer and learning-rate settings. The final selected network is then saved as `models/mnist_classifier.pkl`.

In [ ]:
tuned_model, tuning = tune_neural_network(x_train, y_train)
tuned_predictions = tuned_model.predict(x_test)
tuned_accuracy = accuracy_score(y_test, tuned_predictions)
joblib.dump(tuned_model, MODEL_DIR / 'mnist_classifier.pkl')
print('Selected parameters:', tuning['best_params'])
print(f'Validation accuracy during tuning: {tuning["validation_accuracy"]:.4f}')
print(f'Final test accuracy: {tuned_accuracy:.4f}')
print(classification_report(y_test, tuned_predictions, digits=4))

In [ ]:
# End-to-end demonstration using the saved final model.
final_model = joblib.load(MODEL_DIR / 'mnist_classifier.pkl')
sample_count = min(10, len(x_test))
sample_indices = np.linspace(0, len(x_test) - 1, sample_count, dtype=int)
sample_predictions = final_model.predict(x_test[sample_indices])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, index, prediction in zip(axes.ravel(), sample_indices, sample_predictions):
    image = x_test[index].reshape(28, 28)
    axis.imshow(image, cmap='gray')
    axis.set_title(f'Pred: {prediction} | True: {y_test[index]}')
    axis.axis('off')
plt.suptitle('Sample predictions from the saved final model')
plt.tight_layout()
plt.show()

## Conclusion for the report

Fill the values below after execution and capture the comparison table, confusion matrices, and sample-prediction plot as report screenshots.

- Baseline accuracy: **[fill from output]**
- Neural-network accuracy: **[fill from output]**
- Tuned final accuracy: **[fill from output]**
- Why the selected model performed better: **[explain using the metrics and confusion matrix]**